In [2]:
import random
import torch

In [3]:
class SyntheticRegressionData:
    """합성 회귀 데이터를 생성하고 미니배치로 제공한다."""
    
    def __init__(
        self,
        weights,
        bias,
        noise_std=0.01,
        num_train=1000,
        num_val=1000,
        batch_size=32,
    ):
        # 데이터 생성 및 batch 구성에 필요한 값을 저장한다.
        self.weights = weights
        self.bias = bias
        self.noise_std = noise_std
        self.num_train = num_train
        self.num_val = num_val
        self.batch_size = batch_size
        
        number_of_examples = num_train + num_val
        number_of_features = weights.numel()
        
        # 전체 feature matrix를 생성
        self.X = torch.randn(
            number_of_examples,
            number_of_features,
        )
        
        # 각 sample에 더할 독립적인 Gaussian noise
        noise = (
            torch.randn(number_of_examples, 1)
            * noise_std
        )
        
        # y = Xw + b + epsilon
        self.y = (
            self.X @ weights.reshape(-1, 1)
            + bias
            + noise
        )
        
    def get_dataloader(self, train):
        # 훈련이면 앞의 num_train개 index를 사용한다.
        if train:
            indices = list(range(self.num_train))

            # 훈련할 때는 sample을 무작위 순서로 읽는다.
            random.shuffle(indices)

        # Validation이면 훈련 데이터 다음 index부터 사용한다.
        else:
            indices = list(
                range(
                    self.num_train,
                    self.num_train + self.num_val,
                )
            )
            
        # indices를 batch_size 간격으로 나눈다.
        for start in range(
            0,
            len(indices),
            self.batch_size,
        ):
            # 현재 batch에 해당하는 index만 잘라낸다.
            batch_indices = torch.tensor(
                indices[start : start + self.batch_size]
            )

            # yield는 batch 하나를 반환한 뒤 실행 위치를 기억한다.
            yield (
                self.X[batch_indices],
                self.y[batch_indices],
            )
        
    def train_dataloader(self):
        # 훈련 데이터를 섞어서 반환한다.
        return self.get_dataloader(train=True)

    def val_dataloader(self):
        # Validation 데이터를 정해진 순서로 반환한다.
        return self.get_dataloader(train=False)

In [6]:
true_weights = torch.tensor([2.0, -3.4])
true_bias = 4.2

data = SyntheticRegressionData(
    weights=true_weights,
    bias=true_bias,
    noise_std=0.01,
    num_train=1000,
    num_val=1000,
    batch_size=32,
)

print("Entire X shape:", data.X.shape)
print("Entire y shape:", data.y.shape)

Entire X shape: torch.Size([2000, 2])
Entire y shape: torch.Size([2000, 1])


In [9]:
# train_dataloader()는 generator를 반환한다.
train_loader = data.train_dataloader()

train_iterator = iter(train_loader)
first_X, first_y = next(train_iterator)

print("First X batch shape:", first_X.shape)
print("First y batch shape:", first_y.shape)

assert first_X.shape == (32, 2)
assert first_y.shape == (32, 1)

First X batch shape: torch.Size([32, 2])
First y batch shape: torch.Size([32, 1])


In [10]:
# Cell 5
# 전체 훈련 데이터를 한 번 순회하며
# batch 수와 사용된 sample 수를 확인한다.
number_of_batches = 0
number_of_samples = 0
last_X = None
last_y = None

for batch_X, batch_y in data.train_dataloader():
    number_of_batches += 1
    number_of_samples += batch_X.shape[0]

    last_X = batch_X
    last_y = batch_y

print("Number of batches:", number_of_batches)
print("Number of samples:", number_of_samples)
print("Last X batch shape:", last_X.shape)
print("Last y batch shape:", last_y.shape)

assert number_of_batches == 32
assert number_of_samples == 1000
assert last_X.shape == (8, 2)
assert last_y.shape == (8, 1)

Number of batches: 32
Number of samples: 1000
Last X batch shape: torch.Size([8, 2])
Last y batch shape: torch.Size([8, 1])


In [11]:
# Cell 6
# Validation 데이터는 shuffle하지 않으므로
# 첫 batch가 원본의 X[1000:1032], y[1000:1032]와 같다.
validation_iterator = iter(data.val_dataloader())
validation_X, validation_y = next(validation_iterator)

expected_X = data.X[1000:1032]
expected_y = data.y[1000:1032]

print("Validation X shape:", validation_X.shape)
print("Validation y shape:", validation_y.shape)

assert torch.equal(validation_X, expected_X)
assert torch.equal(validation_y, expected_y)

Validation X shape: torch.Size([32, 2])
Validation y shape: torch.Size([32, 1])
